# Observability with hooks

_Add logging and tracing, then materialize a trajectory into reusable Python._

⏱️ **~20 min** · Part of the [JAZ](https://jaz-lang.ai) tutorial series.

> Run the setup cells once, then work through the rest top to bottom.

## Setup

In [ ]:
# Install JAZ from PyPI
%pip install -q jaz-lang

In [ ]:
import os, getpass

# JAZ uses native OpenAI / Anthropic clients. Set at least one key.
# In Colab you can also use the secrets manager (key icon, left sidebar).
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')


In [ ]:
import jaz

# Pick the model all agents use. Format is 'provider/model'.
# Swap to an Anthropic model (and set ANTHROPIC_API_KEY) if you prefer.
jaz.configure(model_config={'model': 'openai/gpt-4o-mini'})

## Logging

Hooks activate as context managers and propagate to nested invokes. Start with a
logger to see what the agent is doing.

In [ ]:
from jaz.hooks import PrintLogger

with PrintLogger():
    result = jaz.invoke("Calculate 2 + 2", return_type=int)
result

## A custom hook

A hook observes **events** and returns **effects**. Here we nudge the agent to be
concise after a few iterations.

In [ ]:
from jaz.hooks import Hook, Event, Effect, ReplIterationEnter, AddInstructionPrompt

class ConciseHook(Hook):
    """Tell the agent to wrap up after iteration 3."""
    def on_event(self, event: Event) -> list[Effect]:
        if isinstance(event, ReplIterationEnter) and event.iteration > 3:
            return [AddInstructionPrompt("Be concise — you're low on iterations.")]
        return []

with ConciseHook():
    result = jaz.invoke("Solve this step by step: 12 * 11", return_type=int)
result

## Distributed tracing (optional)

Install the tracing extra (`pip install 'jaz-lang[tracing]'`) and set your Langfuse
keys to send traces to Langfuse Cloud.

```python
from jaz.hooks import LangfuseTracingHook

with LangfuseTracingHook():  # needs LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY
    result = jaz.invoke("Calculate 2 + 2", return_type=int)
```

That's the tour! Browse the [reference docs](https://jaz-lang.ai/docs/reference/hooks)
for every event and effect.